# End-to-End Agent Lifecycle Demo

This notebook walks through the **complete lifecycle** of an AI agent in Microsoft Foundry —
first showing each step in the **Portal UI**, then reproducing the exact same step **in code**.

```
Create  →  Test  →  Trace  →  Evaluate  →  Publish  →  Monitor
```

**Use case:** An IT Helpdesk agent that answers employee questions using a knowledge base
(IT policy docs uploaded to a vector store via File Search).

> ⚠️ **Prerequisite:** Run `00-setup.ipynb` first to deploy the Azure infrastructure.
> This notebook is fully self-contained — no portal setup needed beforehand.

In [ ]:
# --- Reconnect to Foundry (run this first) ---
import subprocess, sys, json, shutil
from pathlib import Path

def run_cmd(args, check=True):
    exe = shutil.which(args[0])
    if exe: args = [exe] + args[1:]
    result = subprocess.run(args, capture_output=True, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(result.stderr.strip())
    return result

# Auto-detect pre-provisioned environment (created by optional cell in 00-setup)
config_path = Path.cwd() / "workshop_config.json"
if config_path.exists():
    _cfg = json.loads(config_path.read_text())
    outputs = {k: v for k, v in _cfg.items() if isinstance(v, dict)}
    RESOURCE_GROUP = _cfg.get("resourceGroup", "")
else:
    from datetime import date
    _user_id = run_cmd(["az", "ad", "signed-in-user", "show", "--query", "id", "-o", "tsv"]).stdout.strip()[:6]
    SUFFIX = f"{_user_id}-{date.today().strftime('%m%d')}"
    RESOURCE_GROUP = f"rg-foundry-workshop-{SUFFIX}"
    outputs = json.loads(run_cmd([
        "az", "deployment", "group", "show", "-g", RESOURCE_GROUP, "-n", "main",
        "--query", "properties.outputs", "-o", "json"
    ]).stdout)

PROJECT_ENDPOINT = outputs["projectEndpoint"]["value"]
MODEL_DEPLOYMENT_NAME = outputs["modelDeploymentName"]["value"]
APP_INSIGHTS_CONN_STR = outputs["appInsightsConnectionString"]["value"]

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, FileSearchTool
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
openai_client = project_client.get_openai_client()
print(f"✅ Connected to: {PROJECT_ENDPOINT}")
print(f"   Model: {MODEL_DEPLOYMENT_NAME}")

---
## Step 1: Create

### 🖥️ Portal Walkthrough

1. Open the [Foundry Portal](https://ai.azure.com) → select your project
2. Navigate to **Agents** in the left menu
3. Click **+ New agent**
4. Configure:
   - **Name:** `it-helpdesk`
   - **Model:** `gpt-4.1`
   - **Instructions:** *"You are Contoso's IT Helpdesk assistant. Use the knowledge base to answer IT policy questions. Cite specific policies and SLAs."*
5. Under **Knowledge**, add:
   - Click **+ Add knowledge** → upload `sample_data/it_helpdesk_faq.md` → a vector store / index is created automatically
6. Click **Save** — the agent is created as version 1

### 💻 Same Step in Code

In [ ]:
# --- 1. Create: Upload knowledge base + create agent ---

AGENT_NAME = "e2e-it-helpdesk"

# Upload IT FAQ to a vector store
vector_store = openai_client.vector_stores.create(name="IT-Helpdesk-KB")
faq_path = Path.cwd() / "sample_data" / "it_helpdesk_faq.md"

with faq_path.open("rb") as f:
    openai_client.vector_stores.files.upload_and_poll(
        vector_store_id=vector_store.id, file=f
    )
print(f"📄 Uploaded {faq_path.name} to vector store '{vector_store.name}'")

# Create the IT Helpdesk agent with File Search
helpdesk_agent = project_client.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are Contoso's IT Helpdesk assistant.\n\n"
            "ROLE: Help employees with IT questions and troubleshoot issues.\n\n"
            "RULES:\n"
            "- Always search the knowledge base before answering IT policy questions\n"
            "- Cite specific policies, SLAs, and contact info from the knowledge base\n"
            "- Be friendly, concise, and professional\n\n"
            "FORMAT: Use short paragraphs. Bold key info like SLAs and links."
        ),
        tools=[
            FileSearchTool(vector_store_ids=[vector_store.id]),
        ],
    ),
)
print(f"\n✅ Agent created: {helpdesk_agent.name} (version {helpdesk_agent.version})")
print(f"   Tool: FileSearch (IT knowledge base)")

---
## Step 2: Test

### 🖥️ Portal Walkthrough

1. In the Foundry Portal, open the **it-helpdesk** agent
2. Click **Test** (playground) in the top-right
3. Try these prompts:
   - *"How do I reset my password?"* — should cite the self-service portal and MFA requirement
   - *"What's the process for installing Docker Desktop?"* — should mention Security approval and SLA
   - Follow up with *"Who needs to approve it?"* — should maintain conversation context
4. Observe how the agent retrieves answers from the uploaded knowledge base

### 💻 Same Step in Code

In [ ]:
# --- 2a. Test: Knowledge query ---

response = openai_client.responses.create(
    input="How do I reset my password?",
    extra_body={
        "agent_reference": {"name": helpdesk_agent.name, "type": "agent_reference"}
    },
)
print("🧑 How do I reset my password?\n")
print(f"🤖 {response.output_text}")
print("\n✅ The agent used FileSearch to find the answer in the IT policy knowledge base.")

In [ ]:
# --- 2b. Test: Another knowledge query ---

response2 = openai_client.responses.create(
    input="What is the escalation process for a critical outage affecting 100 users?",
    extra_body={
        "agent_reference": {"name": helpdesk_agent.name, "type": "agent_reference"}
    },
)
print("🧑 What is the escalation process for a critical outage affecting 100 users?\n")
print(f"🤖 {response2.output_text}")

In [ ]:
# --- 2c. Test: Multi-turn conversation (context retention) ---

conversation = openai_client.conversations.create()

# Turn 1
r1 = openai_client.responses.create(
    input="I need to install Docker Desktop on my work laptop. What's the process?",
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": helpdesk_agent.name, "type": "agent_reference"}},
)
print("🧑 Turn 1: I need to install Docker Desktop on my work laptop. What's the process?")
print(f"🤖 {r1.output_text}\n")

# Turn 2 — follow-up referencing previous context
r2 = openai_client.responses.create(
    input="What's the SLA for that? And who needs to approve it?",
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": helpdesk_agent.name, "type": "agent_reference"}},
)
print("🧑 Turn 2: What's the SLA for that? And who needs to approve it?")
print(f"🤖 {r2.output_text}")
print("\n✅ The agent maintained conversation context across turns.")

---
## Step 3: Trace

### 🖥️ Portal Walkthrough

1. In the Foundry Portal, navigate to **Tracing** in the left menu
2. You should see traces for each `responses.create()` call above
3. Click on a trace to inspect:
   - **Spans:** Each step (LLM call, tool invocation, retrieval) is a separate span
   - **Input/Output:** See the exact prompts and responses
   - **Latency:** Time taken for each span
   - **Tool calls:** Which tools were invoked and their results
4. Compare traces for the knowledge query vs. the ticket creation query — notice the different tool spans

> 💡 Tracing is **automatic** — every agent call with `agent_reference` generates server-side traces.
> No extra code needed. The traces flow to Application Insights via the connection configured in setup.

### 💻 Same Step in Code

In [ ]:
# --- 3. Trace: Query Application Insights for agent traces ---
# Foundry agent traces are auto-collected in the AppDependencies table.

from azure.monitor.query import LogsQueryClient
from datetime import timedelta
import pandas as pd

# Get the Log Analytics workspace customer ID (needed by LogsQueryClient)
# Auto-detect: find the first Log Analytics workspace in the resource group
_la_list = run_cmd([
    "az", "monitor", "log-analytics", "workspace", "list",
    "-g", RESOURCE_GROUP,
    "--query", "[0].{name:name, customerId:customerId}", "-o", "json"
], check=False)

if _la_list.returncode != 0 or _la_list.stdout.strip() in ("", "null", "[]"):
    print("⚠️ No Log Analytics workspace found in resource group. Skipping trace query.")
    print("   Run the App Insights deployment in 00-setup.ipynb first.")
    WORKSPACE_ID = None
else:
    _la_info = json.loads(_la_list.stdout)
    WORKSPACE_ID = _la_info["customerId"]
    LOG_ANALYTICS_NAME = _la_info["name"]
    print(f"📊 Log Analytics workspace: {LOG_ANALYTICS_NAME} ({WORKSPACE_ID})")

if WORKSPACE_ID:
    logs_client = LogsQueryClient(credential)

    # Query agent traces from AppDependencies (where Foundry stores them)
    query = """
    AppDependencies
    | where TimeGenerated > ago(1h)
    | extend props = parse_json(Properties)
    | where tostring(props["gen_ai.agent.id"]) has "it-helpdesk"
    | extend
        Operation = tostring(props["gen_ai.operation.name"]),
        AgentId = tostring(props["gen_ai.agent.id"]),
        Model = tostring(props["gen_ai.response.model"]),
        InputTokens = tostring(props["gen_ai.usage.input_tokens"]),
        OutputTokens = tostring(props["gen_ai.usage.output_tokens"]),
        ToolName = tostring(props["gen_ai.tool.name"])
    | project TimeGenerated, Name, Operation, AgentId, Model, InputTokens, OutputTokens, ToolName, DurationMs, Success
    | order by TimeGenerated desc
    | take 15
    """

    result = logs_client.query_workspace(
        WORKSPACE_ID,
        query,
        timespan=timedelta(hours=1),
    )

    table = result.tables[0]
    col_names = [c.name if hasattr(c, 'name') else str(c) for c in table.columns]

    if table.rows:
        df = pd.DataFrame(table.rows, columns=col_names)
        # Format for readability
        if "DurationMs" in df.columns:
            df["DurationMs"] = df["DurationMs"].apply(lambda x: f"{x:.0f}" if x else "")
        if "TimeGenerated" in df.columns:
            df["TimeGenerated"] = df["TimeGenerated"].apply(
                lambda x: x.strftime("%H:%M:%S") if hasattr(x, 'strftime') else str(x)
            )
        print(f"\n✅ Found {len(df)} trace spans:\n")
        print(df.to_string(index=False))
    else:
        print("\n⏳ No traces found yet. Traces can take 2-5 minutes to appear in App Insights.")
        print("   Re-run this cell after a few minutes, or check the Foundry Portal → Tracing.")

---
## Step 4: Evaluate

### 🖥️ Portal Walkthrough

1. In the Foundry Portal, navigate to **Build** -> **Evaluation** in the left menu
2. Click **+ New evaluation** → select your agent -> "Next" -> "Existing dataset"
3. Upload an evaluation dataset (JSONL with `query`, `response`, `context` columns) -> "Next"
4. In Field mapping: select a model as judge, then map the columns (query, ground truth, context) -> "Next"
5. in Configue agents: click "configure", add "user" role, type in "{{item.query}}" and click "save" -> "Next"
6. In Criteria, select the evaluators needed -> "Next"
7. in Review, update the Evaluation name (if needed) and click **Submit** 

### 💻 Same Step in Code

In [ ]:
# --- 4a. Evaluate: Single-response evaluation (cloud) ---
# Run a quick eval on one agent response using the OpenAI Evals API.

from openai.types.eval_create_params import DataSourceConfigCustom
from openai.types.evals.create_eval_jsonl_run_data_source_param import (
    CreateEvalJSONLRunDataSourceParam,
    SourceFileContent,
    SourceFileContentContent,
    SourceFileID,
)
import time

# Get a fresh response from the agent
test_query = "How do I reset my password?"
test_response = openai_client.responses.create(
    input=test_query,
    extra_body={"agent_reference": {"name": helpdesk_agent.name, "type": "agent_reference"}},
).output_text

print(f"🧑 {test_query}")
print(f"🤖 {test_response[:200]}...\n")

# Define a single-row inline evaluation
data_source_config = DataSourceConfigCustom(
    type="custom",
    item_schema={
        "type": "object",
        "properties": {
            "query": {"type": "string"},
            "response": {"type": "string"},
        },
        "required": ["query", "response"],
    },
)

testing_criteria = [
    {
        "type": "azure_ai_evaluator",
        "name": "coherence",
        "evaluator_name": "builtin.coherence",
        "initialization_parameters": {"deployment_name": MODEL_DEPLOYMENT_NAME},
        "data_mapping": {"query": "{{item.query}}", "response": "{{item.response}}"},
    },
    {
        "type": "azure_ai_evaluator",
        "name": "relevance",
        "evaluator_name": "builtin.relevance",
        "initialization_parameters": {"deployment_name": MODEL_DEPLOYMENT_NAME},
        "data_mapping": {"query": "{{item.query}}", "response": "{{item.response}}"},
    },
    {
        "type": "azure_ai_evaluator",
        "name": "fluency",
        "evaluator_name": "builtin.fluency",
        "initialization_parameters": {"deployment_name": MODEL_DEPLOYMENT_NAME},
        "data_mapping": {"query": "{{item.query}}", "response": "{{item.response}}"},
    },
]

eval_obj = openai_client.evals.create(
    name="Helpdesk Single Response Check",
    data_source_config=data_source_config,
    testing_criteria=testing_criteria,
)

# Run with inline data (single row)
single_run = openai_client.evals.runs.create(
    eval_id=eval_obj.id,
    name="single-response-check",
    data_source=CreateEvalJSONLRunDataSourceParam(
        type="jsonl",
        source=SourceFileContent(
            type="file_content",
            content=[SourceFileContentContent(item={"query": test_query, "response": test_response})],
        ),
    ),
)

# Poll for completion
while True:
    single_run = openai_client.evals.runs.retrieve(run_id=single_run.id, eval_id=eval_obj.id)
    if single_run.status in ("completed", "failed"):
        break
    time.sleep(3)

if single_run.status == "completed":
    items = list(openai_client.evals.runs.output_items.list(run_id=single_run.id, eval_id=eval_obj.id))
    print("Evaluating IT Helpdesk response...\n")
    for r in items[0].results:
        print(f"  {r.name}: {r.score:.0f}/5 ({r.label})")
    print("\n✅ Single-response evaluation complete.")
else:
    print(f"❌ Evaluation failed: {single_run.status}")

In [ ]:
# --- 4b. Evaluate: Agent Target Evaluation (new Foundry portal) ---
# The agent generates responses at runtime for each query in the dataset.
# Evaluators then score the live responses against ground truth.

import time
from openai.types.eval_create_params import DataSourceConfigCustom
from openai.types.evals.create_eval_jsonl_run_data_source_param import (
    CreateEvalJSONLRunDataSourceParam,
    SourceFileID,
)

# 1. Upload the evaluation dataset (query + ground_truth + context, no pre-recorded response)
eval_data_path = str(Path.cwd() / "sample_data" / "helpdesk_eval_dataset.jsonl")
data_id = project_client.datasets.upload_file(
    name="helpdesk-eval-data",
    version="2",
    file_path=eval_data_path,
).id
print(f"📄 Dataset uploaded: {data_id}")

# 2. Define the data schema — include_sample_schema=True so evaluators can reference agent output
data_source_config = DataSourceConfigCustom(
    type="custom",
    item_schema={
        "type": "object",
        "properties": {
            "query": {"type": "string"},
            "ground_truth": {"type": "string"},
            "context": {"type": "string"},
        },
        "required": ["query", "ground_truth"],
    },
    include_sample_schema=True,
)

# 3. Define the message template — how queries are sent to the agent
input_messages = {
    "type": "template",
    "template": [
        {
            "type": "message",
            "role": "user",
            "content": {"type": "input_text", "text": "{{item.query}}"},
        }
    ],
}

# 4. Define the agent target — the agent generates responses at runtime
target = {
    "type": "azure_ai_agent",
    "name": AGENT_NAME,
}

# 5. Define evaluators — {{sample.output_text}} is the agent's live response
testing_criteria = [
    {
        "type": "azure_ai_evaluator",
        "name": "coherence",
        "evaluator_name": "builtin.coherence",
        "initialization_parameters": {"deployment_name": MODEL_DEPLOYMENT_NAME},
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{sample.output_text}}",
        },
    },
    {
        "type": "azure_ai_evaluator",
        "name": "relevance",
        "evaluator_name": "builtin.relevance",
        "initialization_parameters": {"deployment_name": MODEL_DEPLOYMENT_NAME},
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{sample.output_text}}",
        },
    },
    {
        "type": "azure_ai_evaluator",
        "name": "groundedness",
        "evaluator_name": "builtin.groundedness",
        "initialization_parameters": {"deployment_name": MODEL_DEPLOYMENT_NAME},
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{sample.output_text}}",
            "context": "{{item.context}}",
        },
    },
    {
        "type": "azure_ai_evaluator",
        "name": "similarity",
        "evaluator_name": "builtin.similarity",
        "initialization_parameters": {"deployment_name": MODEL_DEPLOYMENT_NAME},
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{sample.output_text}}",
            "ground_truth": "{{item.ground_truth}}",
        },
    },
]

# 6. Create the evaluation + run with agent target
eval_object = openai_client.evals.create(
    name="IT Helpdesk Agent Evaluation",
    data_source_config=data_source_config,
    testing_criteria=testing_criteria,
)
print(f"✅ Evaluation created: {eval_object.id}")

eval_run = openai_client.evals.runs.create(
    eval_id=eval_object.id,
    name="helpdesk-agent-eval",
    data_source={
        "type": "azure_ai_target_completions",
        "source": {"type": "file_id", "id": data_id},
        "input_messages": input_messages,
        "target": target,
    },
)
print(f"⏳ Evaluation run started: {eval_run.id}")
print(f"   The agent will generate live responses for each query...\n")

# 7. Poll for completion
while True:
    run = openai_client.evals.runs.retrieve(run_id=eval_run.id, eval_id=eval_object.id)
    if run.status in ("completed", "failed"):
        break
    print("   Waiting for eval run to complete...")
    time.sleep(5)

if run.status == "completed":
    output_items = list(
        openai_client.evals.runs.output_items.list(run_id=run.id, eval_id=eval_object.id)
    )
    print(f"\n=== Results ({len(output_items)} rows) ===")
    for i, item in enumerate(output_items):
        scores = {r.name: f"{r.score:.1f} ({r.label})" for r in item.results}
        print(f"  Row {i+1}: {scores}")

    print(f"\n🔗 View in Foundry Portal: {run.report_url}")
    print("✅ Agent target evaluation complete! The agent generated live responses, scored against ground truth.")
else:
    print(f"❌ Evaluation run failed: {run.status}")
    if hasattr(run, 'error'):
        print(f"   Error: {run.error}")

---
## Step 5: Publish

### What is Publishing?

During development, agents live inside your Foundry project — great for building and testing,
but not for sharing with external consumers. **Publishing** promotes an agent version into a
production-ready **Agent Application** — an independent Azure resource with:

- **Stable endpoint** — the URL stays the same even as you roll out new agent versions
- **Dedicated identity** — its own Entra agent identity, separate from the project identity
- **Independent RBAC** — control who can invoke the agent without giving them project access
- **Azure Policy integration** — governed as an ARM resource

> ⚠️ **Identity changes on publish:** Tools using agent identity authentication switch from the
> project's shared identity to the application's unique identity. You must reassign RBAC
> permissions to the new identity for any resources the agent accesses.

### 🖥️ Portal Walkthrough

1. In the Foundry Portal, open the **it-helpdesk** agent
2. Edit the instructions — refine based on evaluation results
3. Click **Save** — this creates a new immutable version
4. Click **Publish Agent** to create an Agent Application and deployment
5. Configure authentication (defaults to Azure RBAC)
6. Assign permissions: grant **Azure AI User** role on the Agent Application resource to users who need to invoke it
7. After publishing, share the stable endpoint URL or integrate into Teams/M365 Copilot

### 💻 Same Step in Code

In [ ]:
# --- 5a. Publish: Create an improved version, then publish as Agent Application ---
# First, create a refined version based on evaluation feedback.
# Then, publish it as an Agent Application with its own endpoint and identity.
#
# ⚠️ Agent Applications are only available in select regions (e.g. eastus2, eastus, westus).
#    If your Foundry resource is in a different region (e.g. swedencentral), skip this cell and cell 16.

import requests, time, re

# Create an improved agent version with refined instructions
helpdesk_agent_v2 = project_client.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are Contoso's IT Helpdesk assistant.\n\n"
            "ROLE: Help employees with IT questions and troubleshoot issues.\n\n"
            "RULES:\n"
            "- Always search the knowledge base before answering IT policy questions\n"
            "- Cite specific policies, SLAs, and contact info from the knowledge base\n"
            "- If the knowledge base doesn't have the answer, say so clearly\n"
            "- Never fabricate policies or SLAs\n"
            "- Be friendly, concise, and professional\n\n"
            "FORMAT:\n"
            "- Use short paragraphs with bold key info like **SLAs** and **links**\n"
            "- Include step-by-step instructions when explaining procedures\n"
            "- End with a follow-up question to ensure the user's issue is resolved"
        ),
        tools=[
            FileSearchTool(vector_store_ids=[vector_store.id]),
        ],
    ),
)
print(f"✅ Created improved agent version: {helpdesk_agent_v2.name} v{helpdesk_agent_v2.version}")

# --- Publish via ARM REST API ---

# Parse project endpoint to extract account name and project name
# Endpoint format: https://<account>.services.ai.azure.com/api/projects/<project>
endpoint_match = re.match(r"https://([^.]+)\.services\.ai\.azure\.com/api/projects/([^/]+)", PROJECT_ENDPOINT)
assert endpoint_match, f"Could not parse PROJECT_ENDPOINT: {PROJECT_ENDPOINT}"
ACCOUNT_NAME = endpoint_match.group(1)
PROJECT_NAME = endpoint_match.group(2)

# Get subscription ID
SUBSCRIPTION_ID = run_cmd(["az", "account", "show", "--query", "id", "-o", "tsv"]).stdout.strip()

# Get Foundry resource ID (useful for RBAC)
FOUNDRY_RESOURCE_ID = run_cmd([
    "az", "resource", "show",
    "-g", RESOURCE_GROUP,
    "-n", ACCOUNT_NAME,
    "--resource-type", "Microsoft.CognitiveServices/accounts",
    "--query", "id", "-o", "tsv"
]).stdout.strip()

# Get ARM token
arm_token = run_cmd([
    "az", "account", "get-access-token",
    "--resource", "https://management.azure.com",
    "--query", "accessToken", "-o", "tsv"
]).stdout.strip()

headers = {
    "Authorization": f"Bearer {arm_token}",
    "Content-Type": "application/json",
}

# Agent Applications require api-version 2025-10-01-preview or later
API_VERSION = "2025-10-01-preview"
base_url = (
    f"https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}"
    f"/resourceGroups/{RESOURCE_GROUP}"
    f"/providers/Microsoft.CognitiveServices/accounts/{ACCOUNT_NAME}"
    f"/projects/{PROJECT_NAME}"
)

APPLICATION_NAME = "it-helpdesk-app"
DEPLOYMENT_NAME = "it-helpdesk-deploy"

# 1. Create Agent Application
app_url = f"{base_url}/applications/{APPLICATION_NAME}?api-version={API_VERSION}"
app_body = {
    "properties": {
        "agents": [{"agentName": AGENT_NAME}]
    }
}

app_resp = requests.put(app_url, headers=headers, json=app_body)
print(f"\n📦 Create Agent Application: {app_resp.status_code}")
if app_resp.status_code in (200, 201):
    print(f"   Application '{APPLICATION_NAME}' created/updated successfully")
else:
    print(f"   Error: {app_resp.text}")

# 2. Create Agent Deployment
deploy_url = f"{base_url}/applications/{APPLICATION_NAME}/agentdeployments/{DEPLOYMENT_NAME}?api-version={API_VERSION}"
deploy_body = {
    "properties": {
        "displayName": "IT Helpdesk Production Deployment",
        "deploymentType": "Managed",
        "protocols": [
            {
                "protocol": "Responses",
                "version": "1.0"
            }
        ],
        "agents": [
            {
                "agentName": AGENT_NAME,
                "agentVersion": str(helpdesk_agent_v2.version),
            }
        ],
    }
}

deploy_resp = requests.put(deploy_url, headers=headers, json=deploy_body)
print(f"\n🚀 Create Deployment: {deploy_resp.status_code}")
if deploy_resp.status_code in (200, 201):
    print(f"   Deployment '{DEPLOYMENT_NAME}' created/updated successfully")
else:
    print(f"   Error: {deploy_resp.text}")

# 3. Verify deployment is running
time.sleep(5)
verify_resp = requests.get(deploy_url, headers=headers)
if verify_resp.status_code == 200:
    state = verify_resp.json().get("properties", {}).get("state", "unknown")
    print(f"\n✅ Deployment state: {state}")
    if state == "Stopped":
        print("   Starting deployment...")
        start_url = f"{base_url}/applications/{APPLICATION_NAME}/agentdeployments/{DEPLOYMENT_NAME}/start?api-version={API_VERSION}"
        requests.post(start_url, headers=headers)
        print("   Start command sent.")
else:
    print(f"\n⚠️ Could not verify deployment: {verify_resp.status_code}")

print(f"   https://{ACCOUNT_NAME}.services.ai.azure.com/api/projects/{PROJECT_NAME}/applications/{APPLICATION_NAME}/protocols/openai")
print(f"\n📌 Application endpoint:")

In [ ]:
# --- 5b. Publish: Invoke the published Agent Application endpoint ---
# The published endpoint is a stable URL with its own RBAC scope.
# Callers need an access token for https://ai.azure.com and the
# 'Azure AI User' role on the Agent Application resource.
#
# ⚠️ Skip this cell if your region does not support Agent Applications (see cell 15).

# Agent Applications require api-version 2025-10-01-preview or later
API_VERSION = "2025-10-01-preview"

# Discover the actual application name (portal may use agent name directly)
list_url = f"{base_url}/applications?api-version={API_VERSION}"
arm_token = run_cmd([
    "az", "account", "get-access-token",
    "--resource", "https://management.azure.com",
    "--query", "accessToken", "-o", "tsv"
]).stdout.strip()
headers["Authorization"] = f"Bearer {arm_token}"

list_resp = requests.get(list_url, headers=headers)
if list_resp.status_code == 200:
    apps = list_resp.json().get("value", [])
    # Prefer the code-created app, fall back to any app for this agent
    actual_app_name = None
    for app in apps:
        if app["name"] == APPLICATION_NAME:
            actual_app_name = APPLICATION_NAME
            break
    if not actual_app_name:
        for app in apps:
            agents = app.get("properties", {}).get("agents", [])
            if any(a.get("agentName") == AGENT_NAME for a in agents):
                actual_app_name = app["name"]
                break
    if actual_app_name:
        APPLICATION_NAME = actual_app_name
        print(f"📌 Using Agent Application: {APPLICATION_NAME}")
    else:
        print(f"⚠️ No application found for agent '{AGENT_NAME}'. Run cell 15 first.")
else:
    print(f"⚠️ Could not list applications: {list_resp.status_code} {list_resp.text}")

# Grant the current user 'Azure AI User' role on the Agent Application
user_id = run_cmd(["az", "ad", "signed-in-user", "show", "--query", "id", "-o", "tsv"]).stdout.strip()

app_resource_id = (
    f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}"
    f"/providers/Microsoft.CognitiveServices/accounts/{ACCOUNT_NAME}"
    f"/projects/{PROJECT_NAME}/applications/{APPLICATION_NAME}"
)

role_result = run_cmd([
    "az", "role", "assignment", "create",
    "--assignee", user_id,
    "--role", "Azure AI User",
    "--scope", app_resource_id,
], check=False)

stderr = role_result.stderr
if role_result.returncode == 0 or "Conflict" in stderr or "already exists" in stderr:
    print("✅ Azure AI User role assigned (or already exists) on Agent Application")
else:
    print(f"⚠️ Role assignment: {stderr}")

# Get an access token for the Foundry data plane
ai_token = run_cmd([
    "az", "account", "get-access-token",
    "--resource", "https://ai.azure.com",
    "--query", "accessToken", "-o", "tsv"
]).stdout.strip()

# Invoke the published Agent Application endpoint (Responses protocol)
app_endpoint = (
    f"https://{ACCOUNT_NAME}.services.ai.azure.com/api/projects/{PROJECT_NAME}"
    f"/applications/{APPLICATION_NAME}/protocols/openai/responses"
    f"?api-version=2025-11-15-preview"
)

invoke_resp = requests.post(
    app_endpoint,
    headers={
        "Authorization": f"Bearer {ai_token}",
        "Content-Type": "application/json",
    },
    json={"input": "How do I reset my password?"},
)

print(f"\n📡 Invoke Agent Application: {invoke_resp.status_code}")
if invoke_resp.status_code == 200:
    result = invoke_resp.json()
    # Extract text from the Responses API output array
    output_text = result.get("output_text", "")
    if not output_text:
        for item in result.get("output", []):
            if item.get("type") == "message" and item.get("role") == "assistant":
                for block in item.get("content", []):
                    if block.get("type") == "output_text":
                        output_text = block.get("text", "")
                        break
                if output_text:
                    break
    print(f"\n🧑 How do I reset my password?\n")
    print(f"🤖 {output_text}")
    print(f"\n✅ Successfully invoked the published Agent Application!")
    print(f"   This response came through the stable application endpoint,")
    print(f"   not the project endpoint — ready for external consumers.")
else:
    print(f"   Error: {invoke_resp.text}")
    if invoke_resp.status_code == 403:
        print("   💡 Ensure the caller has the 'Azure AI User' role on the Agent Application resource.")
        print("   RBAC assignments can take a few minutes to propagate.")

---
## Step 6: Monitor

### 🖥️ Portal Walkthrough

1. In the Foundry Portal → **Tracing**: See real-time traces for every agent invocation
2. In the Azure Portal → **Application Insights** → **Transaction search**: Full distributed tracing
3. In the Azure Portal → **Application Insights** → **Logs**: Run KQL queries for analytics
4. Useful metrics to monitor:
   - **Invocation count** — how often is the agent used?
   - **Latency** — how fast are responses?
   - **Error rate** — are calls failing?
   - **Tool usage** — which tools are invoked most?

### 💻 Same Step in Code

In [ ]:
# --- 6. Monitor: Query App Insights for agent analytics ---

import pandas as pd
from azure.monitor.query import LogsQueryClient
from datetime import timedelta

if not WORKSPACE_ID:
    print("⚠️ No Log Analytics workspace found. Skipping monitoring queries.")
    print("   Ensure App Insights was deployed in 00-setup.ipynb (optional cell or Section 0A).")
else:
    logs_client = LogsQueryClient(credential)

    # Dashboard query: agent-level metrics from AppDependencies
    dashboard_query = """
    AppDependencies
    | where TimeGenerated > ago(1h)
    | extend props = parse_json(Properties)
    | where tostring(props["gen_ai.agent.id"]) has "it-helpdesk"
    | extend Operation = tostring(props["gen_ai.operation.name"])
    | where Operation == "invoke_agent"
    | summarize
        TotalInvocations = count(),
        AvgLatencyMs = round(avg(DurationMs), 0),
        MaxLatencyMs = round(max(DurationMs), 0),
        FailureCount = countif(Success == false),
        SuccessRate = round(100.0 * countif(Success == true) / count(), 1)
    """

    result = logs_client.query_workspace(
        WORKSPACE_ID,
        dashboard_query,
        timespan=timedelta(hours=1),
    )

    table = result.tables[0]
    col_names = [c.name if hasattr(c, 'name') else str(c) for c in table.columns]

    if table.rows:
        df = pd.DataFrame(table.rows, columns=col_names)
        print("📊 Agent Monitoring Dashboard (Last 1 Hour)\n")
        print(df.to_string(index=False))
    else:
        print("⏳ No invocation data found yet.")

    # Tool usage breakdown
    tool_query = """
    AppDependencies
    | where TimeGenerated > ago(1h)
    | extend props = parse_json(Properties)
    | where tostring(props["gen_ai.agent.id"]) has "it-helpdesk"
    | extend Operation = tostring(props["gen_ai.operation.name"]),
             ToolName = tostring(props["gen_ai.tool.name"])
    | where Operation == "execute_tool"
    | summarize Calls = count(), AvgLatencyMs = round(avg(DurationMs), 0) by ToolName
    """

    result2 = logs_client.query_workspace(WORKSPACE_ID, tool_query, timespan=timedelta(hours=1))
    table2 = result2.tables[0]
    col_names2 = [c.name if hasattr(c, 'name') else str(c) for c in table2.columns]

    if table2.rows:
        df2 = pd.DataFrame(table2.rows, columns=col_names2)
        print("\n\n📊 Tool Usage Breakdown\n")
        print(df2.to_string(index=False))

    print("\n✅ In production, you would set up Azure Monitor alerts on these metrics")
    print("   (e.g., alert if SuccessRate < 95% or AvgLatencyMs > 5000).")

---
## 🎉 Lifecycle Complete!

You've walked through the full AI agent lifecycle in Microsoft Foundry:

| Step | Portal | Code |
|------|--------|------|
| **Create** | New agent + knowledge in Foundry UI | `create_version()` + `FileSearchTool` |
| **Test** | Playground chat | `responses.create()` with `agent_reference` |
| **Trace** | Tracing tab — spans, latency, tool calls | `LogsQueryClient` + KQL on App Insights |
| **Evaluate** | Evaluation tab — scorecards | `evaluate()` with built-in evaluators |
| **Publish** | Publish Agent → Agent Application + deployment | ARM REST API: create application + deployment, invoke via stable endpoint |
| **Monitor** | App Insights dashboards + alerts | KQL queries for invocations, latency, errors |

**Next steps:**
- Add custom function tools (see `02-tools.ipynb` for examples)
- Explore multi-agent orchestration in `04-orchestration.ipynb`
- Build your own agent in `05-byod.ipynb`
- See the [Foundry documentation](https://learn.microsoft.com/azure/ai-foundry/) for advanced topics